In [ ]:
%load_ext autoreload
%autoreload 2
import os
import sys
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = '1'
import numpy as np
import pandas as pd
from typing import Dict, List
# from dotenv import load_dotenv  # pip install python-dotenv
from dynclim.features.safety import SegmentationConfig, ThresholdConfig, CompositeConfig 
from dynclim.experiments.runner import ExperimentRunner
from dynclim.utils import loadDomainsFromYaml
import arrow
# from dynclim.data.database import get_reports_from_db, create_database_engine
# load_dotenv(dotenv_path='/dynClimToolkit/.env')
# load_dotenv()
from pathlib import Path
# If your notebook is in the project root
# project_root = Path("/Users/mrosen44/Documents/dev/DynamicNarrativeClimateToolkit")
project_root = Path("/Users/michaelrosen/Documents/dev/DynamicNarrativeClimateToolkit")
# Option A — explicit reload of the package init
import importlib, dynclim.data
importlib.reload(dynclim.data)

from dynclim.data import (
    MSHADataPipeline, MSHAConfig,
    MSHAOpsPipeline, MSHAOpsConfig,
    MSHACovariatesPipeline, MSHACovariatesConfig,
)
%reload_ext autoreload

In [ ]:
import logging
logging.getLogger().setLevel(logging.INFO)

raw_dir    = project_root / "dynclim/data/raw/msha"
output_dir = project_root / "dynclim/data/processed/msha"

# 1. Exposure / offset (employee-hours + coal tonnage) -> ops_mine_quarter.parquet
ops = MSHAOpsPipeline(raw_dir=raw_dir, output_dir=output_dir)
ops_results = ops.run(save=True, include_yearly=False)
print(ops.summary())

# 2. Operational covariates (Violations + Inspections) + Mines master
cov = MSHACovariatesPipeline(raw_dir=raw_dir, output_dir=output_dir)
cov_results = cov.run(save=True)
print(cov.summary(cov_results))

In [ ]:
# prepare domains and data for feature extraction
domain_definitions = loadDomainsFromYaml(project_root / 'config/domains/safety_domains.yaml')

# 3. Accidents injury+narrative core -> events.parquet (input to feature extraction)
pipeline = MSHADataPipeline(
    raw_dir=raw_dir,
    output_dir=output_dir,
    config=MSHAConfig(include_contractors=False),  # operator-only default (v1)
)
results = pipeline.run(save=True, save_unfiltered=True)
report_data = pipeline.get_report_data()  # [(eid, narrative), ...]
print(f"report_data: {len(report_data):,} narratives")

In [ ]:
# Run experiment with checkpointing

runner = ExperimentRunner(
    domain_definitions = domain_definitions,
    use_acceleration=True,
    device_pref='auto',
    use_parallel=False,
    embedding_model_names=[
        'all-MiniLM-L6-v2',        # small, from sentence transformer team
        # 'all-mpnet-base-v2',     # medium, from sentence transformer team
        'all-roberta-large-v1',  # large, from sentence transformer team
        'BAAI/bge-m3'            # successor to BAAI model above
        ]
)

# Test with subset of configurations
config_filter = {
    #'segmentation_methods': ['sentence'],
    #'threshold_methods': ['top_k'],
    #'composite_methods': ['attention_weight', 'weighted_average'],
    'sentiment_methods': ['vader','roberta','siebert']
}

results_df = runner.run_experiment(
    report_data,
    experiment_name = "msha_06-01-2026",
    config_filter=config_filter,
    resume_from_checkpoint=True,
    # save_interval=2,
    chunk_size=None
    # max_workers=10
)